# FIS Mamdani — Suporte à Decisão Agrícola com Lógica Fuzzy

Sistema de inferência fuzzy (método **Mamdani**) que analisa variáveis climáticas para recomendar janelas de pulverização, monitorar estresse hídrico, sugerir irrigação e estimar produtividade.

## Instalação da dependência

In [ ]:
pip install scikit-fuzzy

## Importação das bibliotecas

- **NumPy** — criação dos universos de discurso (intervalos numéricos das variáveis).
- **scikit-fuzzy** — motor de inferência fuzzy; `control` fornece as abstrações `Antecedent`, `Consequent` e `Rule`.

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

## Variáveis antecedentes (entradas)

Cada antecedente representa uma grandeza climática com seu respectivo universo de discurso:

| Variável | Faixa | Unidade |
|---|---|---|
| Temperatura | 0 – 60 | °C |
| Umidade | 0 – 100 | % |
| Chuva | 0 – 500 | mm |
| Vento | 0 – 150 | km/h |
| Delta T | 0 – 40 | °C |

In [ ]:
temperature = ctrl.Antecedent(np.arange(0, 60, 0.1), 'Temperatura')
humidity = ctrl.Antecedent(np.arange(0, 101, 1.0), 'Umidade')
rain = ctrl.Antecedent(np.arange(0, 500, 0.2), 'Chuva')
wind = ctrl.Antecedent(np.arange(0, 150, 0.5), 'Vento')
delta_t = ctrl.Antecedent(np.arange(0, 40, 0.1), 'Delta T')

## Funções de pertinência dos antecedentes

Cada variável de entrada recebe **7 conjuntos fuzzy** gerados automaticamente via `automf`. Os nomes refletem termos agronômicos para facilitar a leitura das regras:

- **Temperatura**: frio_extremo, frio, ameno, **ideal**, quente, estresse_termico, critico
- **Umidade**: deserto, critico_seco, seco, **ideal**, umido, saturado, condensacao
- **Chuva**: seco, trace, leve, moderada, forte, muito_forte, extrema
- **Vento**: calmo, ideal_pulv, moderado, atencao_pulv, forte, muito_forte, tempestade
- **Delta T**: inversao_termica, ideal_pulv, aceitavel, atencao, critico_pulv, proibido, extremo

In [ ]:
temperature.automf(
    number=7,
    names=['frio_extremo', 'frio', 'ameno', 'ideal', 'quente', 'estresse_termico', 'critico']
)
humidity.automf(
    number=7,
    names=['deserto', 'critico_seco', 'seco', 'ideal', 'umido', 'saturado', 'condensacao']
)
rain.automf(
    number=7,
    names=['seco', 'trace', 'leve', 'moderada', 'forte', 'muito_forte', 'extrema']
)
wind.automf(
    number=7,
    names=['calmo', 'ideal_pulv', 'moderado', 'atencao_pulv', 'forte', 'muito_forte', 'tempestade']
)
delta_t.automf(
    number=7,
    names=['inversao_termica', 'ideal_pulv', 'aceitavel', 'atencao', 'critico_pulv', 'proibido', 'extremo']
)

## Variáveis consequentes (saídas)

| Consequente | Universo | Descrição |
|---|---|---|
| Recomendação de Pulverização (`sp`) | 0 – 10 | Classifica em **proibida**, **atenção** ou **janela disponível** (funções triangulares manuais) |
| Estresse Hídrico (`wh`) | 0 – 1 | Índice normalizado de estresse hídrico da cultura |
| Recomendação de Irrigação (`ir`) | 0 – 10 | Nível de urgência para irrigação |
| Produtividade Estimada (`bp`) | 0 – 100 | Saída inferida por regras fuzzy dedicadas à produtividade; a regra combinada, quando presente, é mantida apenas como legado |

In [ ]:
spray_recommendation = ctrl.Consequent(np.arange(0, 11, 1), 'sp')
spray_recommendation['proibida'] = fuzz.trimf(spray_recommendation.universe, [0, 0, 4])
spray_recommendation['atencao'] = fuzz.trimf(spray_recommendation.universe, [3, 5, 7])
spray_recommendation['janela_disponivel'] = fuzz.trimf(spray_recommendation.universe, [6, 10, 10])

water_stress = ctrl.Consequent(np.arange(0, 1.1, 0.1), 'wh')
water_stress['baixo'] = fuzz.trimf(water_stress.universe, [0, 0.2, 1])
water_stress['medio'] = fuzz.trimf(water_stress.universe, [0, 0.5, 1])
water_stress['alto'] = fuzz.trimf(water_stress.universe, [0.5, 0.8, 1])

irr_recommendation = ctrl.Consequent(np.arange(0, 11, 1), 'ir')
irr_recommendation['desnecessaria'] = fuzz.trimf(irr_recommendation.universe, [0, 3, 4])
irr_recommendation['opcional'] = fuzz.trimf(irr_recommendation.universe, [0, 5, 6])
irr_recommendation['recomendada'] = fuzz.trimf(irr_recommendation.universe, [5, 9, 10])

bet_productivity = ctrl.Consequent(np.arange(0, 101, 1), 'bp')
bet_productivity['baixa'] = fuzz.trimf(bet_productivity.universe, [0, 10, 40])
bet_productivity['medio'] = fuzz.trimf(bet_productivity.universe, [0, 50, 70])
bet_productivity['alta'] = fuzz.trimf(bet_productivity.universe, [50, 80, 100])

## Regras fuzzy — Recomendação de Pulverização

Regras do tipo **SE-ENTÃO** que combinam as condições climáticas para classificar a pulverização:

- **Janela disponível** — Delta T ideal, sem chuva significativa, umidade e temperatura ideais.
- **Atenção** — condições parcialmente favoráveis (Delta T aceitável, chuva leve, vento moderado, temperatura amena/quente, ou umidade muito baixa).
- **Proibida** — condições adversas (Delta T crítico/extremo, chuva forte, vento forte/tempestade, temperatura extrema, ou umidade saturada/condensação).

In [ ]:
sp_janela_r1 = ctrl.Rule(
    delta_t['ideal_pulv']
    & (rain['seco'] | rain['trace'])
    & humidity['ideal']
    & temperature['ideal'],
    spray_recommendation['janela_disponivel']
)

sp_atencao_r1 = ctrl.Rule(
    (delta_t['aceitavel'] | delta_t['atencao'])
    & rain['trace']
    & (humidity['seco'] | humidity['umido'])
    & (temperature['ameno'] | temperature['quente']),
    spray_recommendation['atencao']
)

sp_atencao_r2 = ctrl.Rule(
    wind['atencao_pulv']
    & (rain['seco'] | rain['trace'])
    & humidity['ideal']
    & temperature['ideal'],
    spray_recommendation['atencao']
)

sp_atencao_r3 = ctrl.Rule(
    rain['leve']
    & (delta_t['ideal_pulv'] | delta_t['aceitavel'])
    & (wind['calmo'] | wind['ideal_pulv'] | wind['moderado']),
    spray_recommendation['atencao']
)

sp_atencao_r4 = ctrl.Rule(
    temperature['frio']
    & (delta_t['ideal_pulv'] | delta_t['aceitavel'])
    & (rain['seco'] | rain['trace']),
    spray_recommendation['atencao']
)

sp_atencao_r5 = ctrl.Rule(
    humidity['deserto'] | humidity['critico_seco'],
    spray_recommendation['atencao']
)

sp_proibida_r1 = ctrl.Rule(
    (delta_t['critico_pulv'] | delta_t['proibido'])
    & (rain['forte'] | rain['muito_forte'] | rain['extrema'] | rain['moderada']),
    spray_recommendation['proibida']
)

sp_proibida_r2 = ctrl.Rule(
    delta_t['inversao_termica'] | delta_t['extremo'],
    spray_recommendation['proibida']
)

sp_proibida_r3 = ctrl.Rule(
    wind['tempestade'] | wind['forte'] | wind['muito_forte'],
    spray_recommendation['proibida']
)

sp_proibida_r4 = ctrl.Rule(
    temperature['frio_extremo'] | temperature['critico'] | temperature['estresse_termico'],
    spray_recommendation['proibida']
)

sp_proibida_r5 = ctrl.Rule(
    humidity['saturado'] | humidity['condensacao'],
    spray_recommendation['proibida']
)

## Task 4 — Estresse hídrico e irrigação (pacote `fuzzylab`)

As regras dos consequentes **`wh`** (estresse hídrico) e **`ir`** (irrigação), com encadeamento *water_stress* → *irr_recommendation*, estão no módulo único `src/fuzzylab/fis/mamdani.py`, junto com as regras de produtividade e a regra combinada legada. A célula seguinte adiciona `src/` ao path (se ainda não usaste `pip install -e .`), importa a interface pública (`build_system` / `run_inference`) e executa três cenários climáticos.

In [ ]:
import sys
from pathlib import Path

# Adiciona src/ ao path se o pacote não estiver instalado (pip install -e .)
_root = Path.cwd()
for base in (_root, _root.parent):
    if (base / "src" / "fuzzylab").is_dir():
        sys.path.insert(0, str(base / "src"))
        break

from fuzzylab.fis import build_system, run_inference

cenarios = {
    "seco": {
        "Temperatura": 38.0,
        "Umidade": 12.0,
        "Chuva": 0.0,
        "Vento": 10.0,
        "Delta T": 8.0,
    },
    "normal": {
        "Temperatura": 26.0,
        "Umidade": 55.0,
        "Chuva": 8.0,
        "Vento": 12.0,
        "Delta T": 8.0,
    },
    "chuvoso": {
        "Temperatura": 22.0,
        "Umidade": 88.0,
        "Chuva": 150.0,
        "Vento": 15.0,
        "Delta T": 7.0,
    },
}

for nome, entradas in cenarios.items():
    system = build_system()
    outputs = run_inference(system, entradas)
    print(nome, outputs)

### Chaves em `fis_simulation.output`

| Chave | Significado |
|-------|-------------|
| `wh` | Estresse hídrico (0–1) |
| `ir` | Recomendação de irrigação (0–10) |
| `sp` | Pulverização |
| `bp` | Produtividade estimada (regra de referência) |

In [ ]:
# Simulação canônica: célula anterior (usa a interface pública fuzzylab.fis).

In [ ]:
# Exemplo pontual (mesmo motor que os cenários):
_system = build_system()
_outputs = run_inference(
    _system,
    {
        "Temperatura": 28,
        "Umidade": 60,
        "Chuva": 0,
        "Vento": 10,
        "Delta T": 8,
    },
)
print("exemplo", _outputs)

## Task 6 — Visualizações das Membership Functions

Plots das funções de pertinência de todos os antecedentes e consequentes para validação visual dos universos de discurso.

In [ ]:
import matplotlib.pyplot as plt

# Importa definições do módulo flat fuzzylab.fis.mamdani
from fuzzylab.fis.mamdani import (
    temperature, humidity, rain, wind, delta_t,
    spray_recommendation, water_stress, irr_recommendation, bet_productivity,
)

# Diretório de artefatos para os plots
FIGURES_DIR = Path.cwd() / "figures"
if not FIGURES_DIR.is_dir():
    FIGURES_DIR = Path.cwd() / "notebooks" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Plot dos antecedentes (entradas)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Membership Functions — Antecedentes (Entradas)', fontsize=14)

antecedentes = [
    (temperature, 'Temperatura (°C)', axes[0, 0]),
    (humidity, 'Umidade (%)', axes[0, 1]),
    (rain, 'Chuva (mm)', axes[0, 2]),
    (wind, 'Vento (km/h)', axes[1, 0]),
    (delta_t, 'Delta T (°C)', axes[1, 1]),
]

for var, titulo, ax in antecedentes:
    for term_name in var.terms:
        ax.plot(var.universe, var[term_name].mf, label=term_name)
    ax.set_title(titulo)
    ax.legend(loc='upper right', fontsize=7)
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_antecedentes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot dos consequentes (saídas)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Membership Functions — Consequentes (Saídas)', fontsize=14)

consequentes = [
    (spray_recommendation, 'Pulverização (sp)', axes[0, 0]),
    (water_stress, 'Estresse Hídrico (wh)', axes[0, 1]),
    (irr_recommendation, 'Irrigação (ir)', axes[1, 0]),
    (bet_productivity, 'Produtividade (bp)', axes[1, 1]),
]

for var, titulo, ax in consequentes:
    for term_name in var.terms:
        ax.plot(var.universe, var[term_name].mf, label=term_name, linewidth=2)
    ax.set_title(titulo)
    ax.legend(loc='best', fontsize=9)
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mf_consequentes.png', dpi=150, bbox_inches='tight')
plt.show()

## Superfícies de Controle 3D

Visualização do comportamento do FIS para pares de variáveis, mantendo as demais constantes. Permite identificar inconsistências nas regras e validar a coerência agronômica das saídas.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def compute_surface(x_range, y_range, x_name, y_name, output_key, fixed_inputs):
    """Calcula superfície de controle para um par de variáveis."""
    Z = np.zeros((len(y_range), len(x_range)))

    for i, y_val in enumerate(y_range):
        for j, x_val in enumerate(x_range):
            sim = build_system()
            inputs = {x_name: x_val, y_name: y_val, **fixed_inputs}
            try:
                out = run_inference(sim, inputs)
                Z[i, j] = out.get(output_key, np.nan)
            except Exception:
                Z[i, j] = np.nan
    return Z

# Superfície 1: Temperatura x Umidade → Estresse Hídrico (wh)
temp_range = np.linspace(5, 50, 25)
hum_range = np.linspace(5, 95, 25)
fixed_1 = {"Chuva": 10.0, "Vento": 10.0, "Delta T": 8.0}

print("Calculando superfície Temperatura x Umidade → Estresse Hídrico...")
Z_wh = compute_surface(temp_range, hum_range, "Temperatura", "Umidade", "wh", fixed_1)

X, Y = np.meshgrid(temp_range, hum_range)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z_wh, cmap='RdYlGn_r', edgecolor='none', alpha=0.9)
ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('Umidade (%)')
ax.set_zlabel('Estresse Hídrico (wh)')
ax.set_title('Superfície de Controle: Temperatura x Umidade → wh\n(Chuva=10mm, Vento=10km/h, ΔT=8°C)')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='wh')
plt.savefig(FIGURES_DIR / 'surface_temp_hum_wh.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Superfície 2: Umidade x Chuva → Produtividade (bp)
hum_range_2 = np.linspace(10, 90, 25)
rain_range = np.linspace(0, 300, 25)
fixed_2 = {"Temperatura": 25.0, "Vento": 10.0, "Delta T": 8.0}

print("Calculando superfície Umidade x Chuva → Produtividade...")
Z_bp = compute_surface(hum_range_2, rain_range, "Umidade", "Chuva", "bp", fixed_2)

X2, Y2 = np.meshgrid(hum_range_2, rain_range)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X2, Y2, Z_bp, cmap='YlGn', edgecolor='none', alpha=0.9)
ax.set_xlabel('Umidade (%)')
ax.set_ylabel('Chuva (mm)')
ax.set_zlabel('Produtividade (bp)')
ax.set_title('Superfície de Controle: Umidade x Chuva → bp\n(Temperatura=25°C, Vento=10km/h, ΔT=8°C)')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='bp')
plt.savefig(FIGURES_DIR / 'surface_hum_rain_bp.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Visualizações salvas em: {FIGURES_DIR}")